# FDM/PLA Defect Detection — Faster R-CNN Training & Inference

Thesis: *Defect Detection in FDM 3D-Printed PLA Specimens Using Deep Learning*

This notebook is a **research prototype** for image-based detection of defects in FDM 3D-printed specimens. It:
1. Loads a Pascal VOC-annotated dataset (exported from Roboflow) with 4 defect classes: **cracking, layer shifting, stringing, warping**.
2. Fine-tunes a **Faster R-CNN** detector with a **ResNet-50 FPN V2** backbone pretrained on COCO.
3. Uses a train/validation/test split and reports detection metrics on the test set.
4. Runs inference that outputs **"No defect"** or the detected defect type(s).

> **Research status:** This is a preliminary prototype. Dataset quality, split independence, class balance, and evaluation should be verified before treating the reported metrics as final thesis results.

> **Before running:** update `CLASS_NAMES` if your XML files use different label spellings.


## 1. Setup

In [ ]:
# Run this first. Colab already has PyTorch + torchvision preinstalled.
!pip install -q torchmetrics pycocotools

import os, glob, xml.etree.ElementTree as ET
import torch, torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from PIL import Image
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Get your dataset into Colab
Upload the Roboflow export (zip) and unzip it, OR mount Google Drive if you saved it there.
Roboflow's Pascal VOC export normally creates this structure:

```
dataset/
  train/   <- images + matching .xml files, together
  valid/
  test/
```

Adjust `DATASET_ROOT` below to match wherever your unzipped folder ends up.


In [ ]:
# --- Option A: upload your 4 archive zips one at a time ---
# Run this cell 4 times (once per archive), changing the extraction folder name each time
# to match what you listed in SOURCE_FOLDERS in the next section.
from google.colab import files
uploaded = files.upload()   # choose one archive .zip when prompted

import zipfile
zip_name = list(uploaded.keys())[0]
extract_to = "archive_cracking"  # <-- CHANGE this each of the 4 times you run this cell
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_to)

print(f"Extracted to: {extract_to}")
print(os.listdir(extract_to))


In [ ]:
# --- Option B: mount Google Drive instead (use this OR the cell above, not both) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_ROOT = "/content/drive/MyDrive/your_dataset_folder"


## 2b. Merge your 4 per-class archives into one dataset
Since you annotated each defect class separately in Roboflow, you exported 4 separate
archives (one per class), each already split into its own `train/valid/test`. This cell
merges all 4 into a single unified `merged_dataset/train`, `merged_dataset/valid`,
`merged_dataset/test`, which is what the rest of this notebook expects.

**Before running:** upload/unzip your 4 archives first (repeat the upload cell above, or
unzip each with `!unzip archive1.zip -d archive1` etc.), then list their folder names in
`SOURCE_FOLDERS` below. If you mounted Drive instead, point these at your Drive paths.


In [ ]:
import shutil

# EDIT THIS: the 4 unzipped archive folder names/paths, one per defect class.
# Each of these must itself contain train/, valid/, test/ subfolders.
SOURCE_FOLDERS = [
    "archive_cracking",
    "archive_layer_shifting",
    "archive_stringing",
    "archive_warping",
]

MERGED_ROOT = "merged_dataset"
SPLITS = ["train", "valid", "test"]

for split in SPLITS:
    os.makedirs(os.path.join(MERGED_ROOT, split), exist_ok=True)

def merge_archives(source_folders, merged_root, splits):
    copied_count = {s: 0 for s in splits}
    for src_folder in source_folders:
        for split in splits:
            src_split_dir = os.path.join(src_folder, split)
            if not os.path.isdir(src_split_dir):
                print(f"  (skipping missing: {src_split_dir})")
                continue

            dst_split_dir = os.path.join(merged_root, split)
            # group files by basename so image+xml pairs move together
            files_by_base = {}
            for fname in os.listdir(src_split_dir):
                base, ext = os.path.splitext(fname)
                files_by_base.setdefault(base, []).append(fname)

            for base, files in files_by_base.items():
                # prefix with source folder name to avoid filename collisions
                # across the 4 archives (Roboflow filenames can repeat)
                prefix = os.path.basename(src_folder.rstrip("/"))
                for fname in files:
                    src_path = os.path.join(src_split_dir, fname)
                    ext = os.path.splitext(fname)[1]
                    new_name = f"{prefix}__{base}{ext}"
                    dst_path = os.path.join(dst_split_dir, new_name)
                    shutil.copy2(src_path, dst_path)
                copied_count[split] += 1
    return copied_count

counts = merge_archives(SOURCE_FOLDERS, MERGED_ROOT, SPLITS)
print("Merged image counts per split:", counts)

# Point the rest of the notebook at the merged dataset
DATASET_ROOT = MERGED_ROOT
print("DATASET_ROOT set to:", DATASET_ROOT)
print(os.listdir(DATASET_ROOT))


## 2c. Dataset audit before training
This check looks for **exact duplicate image files across train/valid/test**. Duplicate images across splits can cause data leakage and artificially inflate test performance.

This is only an exact-file check; visually similar photos of the same specimen may still require manual inspection.

In [ ]:
import hashlib

def file_sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def audit_split_duplicates(dataset_root):
    hashes = {}
    exts = ('.jpg', '.jpeg', '.png')
    for split in ['train', 'valid', 'test']:
        split_dir = os.path.join(dataset_root, split)
        for fname in os.listdir(split_dir):
            if fname.lower().endswith(exts):
                path = os.path.join(split_dir, fname)
                hashes.setdefault(file_sha256(path), []).append((split, fname))

    duplicates = [items for items in hashes.values() if len({x[0] for x in items}) > 1]
    print(f'Exact duplicate image groups across different splits: {len(duplicates)}')
    if duplicates:
        print('WARNING: inspect these groups before reporting final test metrics:')
        for group in duplicates[:20]:
            print(group)
    else:
        print('No exact duplicate image files were found across train/valid/test.')
    return duplicates

duplicate_groups = audit_split_duplicates(DATASET_ROOT)


## 3. Class labels

In [ ]:
# EDIT THIS if your Roboflow class names differ (check the <name> tag inside any .xml file).
# Index 0 is always reserved for "background" -- torchvision's Faster R-CNN requires this.
CLASS_NAMES = ["background", "cracking", "layer shifting", "stringing", "warping"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)  # 4 defects + background = 5
print(CLASS_TO_IDX)


## 4. Dataset class (parses Pascal VOC XML, with data augmentation)
The `train=True` flag turns on light augmentation (random horizontal flip + color jitter) for
the training set only — this exposes the model to more visual variety without changing what
the defects actually look like, which helps combat overfitting on a modest-sized dataset.
Validation and test sets use `train=False` so they stay unaugmented and evaluation stays fair.


In [ ]:
import random
import torchvision.transforms.functional as TF

class VOCDefectDataset(Dataset):
    """
    Expects a folder containing image files (.jpg/.png) each paired with a
    same-name .xml Pascal VOC annotation file (Roboflow's default export layout).
    """
    def __init__(self, folder, class_to_idx, train=False):
        self.folder = folder
        self.class_to_idx = class_to_idx
        self.train = train  # enables augmentation when True (use only for the training set)
        exts = (".jpg", ".jpeg", ".png")
        self.image_paths = sorted([
            p for p in glob.glob(os.path.join(folder, "*"))
            if p.lower().endswith(exts)
        ])

    def __len__(self):
        return len(self.image_paths)

    def _xml_path(self, image_path):
        base = os.path.splitext(image_path)[0]
        return base + ".xml"

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        xml_path = self._xml_path(img_path)

        img = Image.open(img_path).convert("RGB")
        width, height = img.size
        boxes, labels = [], []

        if os.path.exists(xml_path):
            tree = ET.parse(xml_path)
            root = tree.getroot()
            for obj in root.findall("object"):
                name = obj.find("name").text.strip()
                if name not in self.class_to_idx:
                    continue
                bnd = obj.find("bndbox")
                xmin = float(bnd.find("xmin").text)
                ymin = float(bnd.find("ymin").text)
                xmax = float(bnd.find("xmax").text)
                ymax = float(bnd.find("ymax").text)
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(self.class_to_idx[name])

        # --- Data augmentation (training set only) ---
        if self.train:
            # Random horizontal flip (p=0.5) -- boxes must be flipped along with the image
            if random.random() < 0.5:
                img = TF.hflip(img)
                if len(boxes) > 0:
                    boxes = [[width - xmax, ymin, width - xmin, ymax] for xmin, ymin, xmax, ymax in boxes]

            # Mild color jitter -- helps the model generalize across lighting/camera differences
            img = TF.adjust_brightness(img, random.uniform(0.8, 1.2))
            img = TF.adjust_contrast(img, random.uniform(0.8, 1.2))

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) else torch.zeros((0,))
        iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd,
        }

        img = F.to_tensor(img)
        return img, target


def collate_fn(batch):
    return tuple(zip(*batch))


## 5. Build the train / valid / test datasets and loaders

In [ ]:
train_dataset = VOCDefectDataset(os.path.join(DATASET_ROOT, "train"), CLASS_TO_IDX, train=True)
valid_dataset = VOCDefectDataset(os.path.join(DATASET_ROOT, "valid"), CLASS_TO_IDX, train=False)
test_dataset  = VOCDefectDataset(os.path.join(DATASET_ROOT, "test"),  CLASS_TO_IDX, train=False)

print("Train images:", len(train_dataset))
print("Valid images:", len(valid_dataset))
print("Test images:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)


## 5b. Dataset summary
Before training, inspect the number of images and bounding boxes in each split. This helps identify class imbalance and unusually small classes.

In [ ]:
from collections import Counter

def count_annotations(dataset):
    counts = Counter()
    total_boxes = 0
    for i in range(len(dataset)):
        _, target = dataset[i]
        for label in target['labels'].tolist():
            counts[CLASS_NAMES[label]] += 1
            total_boxes += 1
    return counts, total_boxes

for split_name, dataset in [('train', train_dataset), ('valid', valid_dataset), ('test', test_dataset)]:
    counts, total = count_annotations(dataset)
    print(f'{split_name}: {len(dataset)} images, {total} annotated boxes')
    print(dict(counts))


## 6. Build the Faster R-CNN model (transfer learning)

In [ ]:
def build_model(num_classes):
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn_v2(weights=weights)

    # Replace the pretrained (COCO, 91-class) prediction head with one sized
    # for our 5 classes (4 defects + background)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = build_model(NUM_CLASSES)
model.to(device)
print(model.roi_heads.box_predictor)


## 7. Training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
    return total_loss / len(loader)


@torch.no_grad()
def validate_loss(model, loader, device):
    # NOTE: torchvision's Faster R-CNN only returns a loss dict in train() mode,
    # so we temporarily keep it in train mode just for the forward pass but
    # under no_grad(), which is the standard workaround for tracking val loss.
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()
    return total_loss / len(loader)


In [ ]:
# --- Checkpoint location: saves to Google Drive so it survives runtime disconnects ---
# Requires Drive to already be mounted (Section 2, Option B). If you haven't mounted
# Drive, uncomment and run this first:
# from google.colab import drive
# drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/FDM_Dataset/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "fasterrcnn_fdm_defects_best.pth")
print("Checkpoints will be saved to:", BEST_CHECKPOINT_PATH)


## 7a. Training and validation loss curves
These curves are saved as PNG files in `results/` so they can be included in the GitHub repository and thesis report.

The lowest validation-loss epoch is marked automatically.

In [ ]:
import matplotlib.pyplot as plt

RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
epochs = range(1, len(history['train_loss']) + 1)
best_valid_idx = int(np.argmin(history['valid_loss']))
best_valid_epoch = best_valid_idx + 1

plt.figure(figsize=(8, 5))
plt.plot(epochs, history['train_loss'], marker='o', label='Training loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss vs Epoch')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_loss.png'), dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history['valid_loss'], marker='o', label='Validation loss')
plt.axvline(best_valid_epoch, linestyle='--', label=f'Lowest validation loss: epoch {best_valid_epoch}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Validation Loss vs Epoch')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'validation_loss.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f'Lowest validation loss: {history["valid_loss"][best_valid_idx]:.4f} at epoch {best_valid_epoch}')


In [ ]:
NUM_EPOCHS = 30  # more epochs is fine now since we save the BEST checkpoint, not just the last one

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# ReduceLROnPlateau lowers the learning rate only when validation loss stops improving,
# instead of on a fixed timer -- this adapts to your actual training dynamics.
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

history = {"train_loss": [], "valid_loss": []}
best_valid_loss = float("inf")
best_epoch = -1
PATIENCE = 7          # stop early if no improvement for this many epochs
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    valid_loss = validate_loss(model, valid_loader, device)
    lr_scheduler.step(valid_loss)

    history["train_loss"].append(train_loss)
    history["valid_loss"].append(valid_loss)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | train_loss: {train_loss:.4f} | valid_loss: {valid_loss:.4f}")

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        torch.save(model.state_dict(), BEST_CHECKPOINT_PATH)
        print(f"  -> New best model (valid_loss={valid_loss:.4f}), saved to Drive.")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"No improvement for {PATIENCE} epochs -- stopping early at epoch {epoch+1}.")
            break

print(f"\nBest model was from epoch {best_epoch} with valid_loss={best_valid_loss:.4f}")

# Reload the BEST checkpoint (not necessarily the last epoch trained) before evaluating
model.load_state_dict(torch.load(BEST_CHECKPOINT_PATH))
print("Loaded best checkpoint for evaluation.")


## 7b. Resuming later without retraining
If your Colab runtime ever disconnects **after** training finished, you don't need to
retrain — as long as this used the Drive-based checkpoint path above. Just re-run
Sections 1, 3, and 6 (setup, class labels, and model-building) to get a fresh `model`
object and re-mount Drive, then run this cell to reload your saved weights directly:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/FDM_Dataset/checkpoints"
BEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "fasterrcnn_fdm_defects_best.pth")

model.load_state_dict(torch.load(BEST_CHECKPOINT_PATH))
model.to(device)
model.eval()
print("Reloaded trained model from Drive -- ready for evaluation or inference, no retraining needed.")


## 8. Evaluation on the test set
This section reports the main detection metrics used in this prototype:
Precision, Recall, F1-Score, mAP@0.5, mAP@0.5:0.95, and a confusion matrix. The mAP metrics
come from `torchmetrics`' COCO-style implementation. Precision/Recall/F1 and the confusion
matrix need a separate calculation, since those depend on matching predicted boxes to ground
truth boxes via IoU (Intersection over Union) — which is a different computation than mAP.

Note: the current dataset uses a 70-10-20 train/valid/test split. The exact split and dataset construction should be reported transparently in the thesis.


In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

@torch.no_grad()
def evaluate_map(model, loader, device, score_threshold=0.5):
    model.eval()
    metric = MeanAveragePrecision(class_metrics=True)

    for images, targets in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        preds, gts = [], []
        for out, tgt in zip(outputs, targets):
            keep = out["scores"] >= score_threshold
            preds.append({
                "boxes": out["boxes"][keep].cpu(),
                "scores": out["scores"][keep].cpu(),
                "labels": out["labels"][keep].cpu(),
            })
            gts.append({
                "boxes": tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu(),
            })
        metric.update(preds, gts)

    return metric.compute()

results = evaluate_map(model, test_loader, device)
print("mAP@0.5:0.95:", results["map"].item())
print("mAP@0.5:     ", results["map_50"].item())
print("mAP@0.75:    ", results["map_75"].item())
print("Per-class AP:", results["map_per_class"])


## 8b. Precision, Recall, F1-Score, and Confusion Matrix
This computes Precision, Recall, and F1-Score using explicit TP/FP/FN matching: a prediction counts as a True Positive (TP) if its box
overlaps a ground-truth box of the same class by at least `iou_threshold` (0.5); otherwise it's a False Positive (FP). Any ground-truth box left unmatched
counts as a False Negative (FN). It also builds a confusion matrix across all 4 classes
plus "background" (covering missed detections and false alarms), with background used to represent missed detections and false alarms.


In [ ]:
from torchvision.ops import box_iou
from collections import defaultdict
import numpy as np

@torch.no_grad()
def evaluate_precision_recall_f1(model, loader, device, iou_threshold=0.5, score_threshold=0.5, num_classes=NUM_CLASSES):
    model.eval()
    TP = defaultdict(int)
    FP = defaultdict(int)
    FN = defaultdict(int)

    # For the confusion matrix: 0 = background, 1..N-1 = defect classes
    y_true_for_cm = []
    y_pred_for_cm = []

    for images, targets in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for output, target in zip(outputs, targets):
            keep = output["scores"] >= score_threshold
            pred_boxes = output["boxes"][keep].cpu()
            pred_labels = output["labels"][keep].cpu()

            gt_boxes = target["boxes"].cpu()
            gt_labels = target["labels"].cpu()

            matched_gt_indices = set()

            for i in range(len(pred_boxes)):
                pl = pred_labels[i].item()
                if len(gt_boxes) == 0:
                    FP[pl] += 1
                    y_true_for_cm.append(0)   # nothing there -> background
                    y_pred_for_cm.append(pl)
                    continue

                ious = box_iou(pred_boxes[i].unsqueeze(0), gt_boxes)[0]
                best_iou, best_idx = ious.max(0)
                best_idx = best_idx.item()

                if best_iou.item() >= iou_threshold and gt_labels[best_idx].item() == pl and best_idx not in matched_gt_indices:
                    TP[pl] += 1
                    matched_gt_indices.add(best_idx)
                    y_true_for_cm.append(pl)
                    y_pred_for_cm.append(pl)
                else:
                    FP[pl] += 1
                    y_true_for_cm.append(0)
                    y_pred_for_cm.append(pl)

            for idx in range(len(gt_labels)):
                if idx not in matched_gt_indices:
                    gl = gt_labels[idx].item()
                    FN[gl] += 1
                    y_true_for_cm.append(gl)
                    y_pred_for_cm.append(0)   # model missed it -> predicted background

    per_class = {}
    for c in range(1, num_classes):
        tp, fp, fn = TP[c], FP[c], FN[c]
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        per_class[c] = {"precision": precision, "recall": recall, "f1": f1, "TP": tp, "FP": fp, "FN": fn}

    total_tp = sum(TP[c] for c in range(1, num_classes))
    total_fp = sum(FP[c] for c in range(1, num_classes))
    total_fn = sum(FN[c] for c in range(1, num_classes))
    overall_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    overall_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    overall_f1 = 2 * overall_precision * overall_recall / (overall_precision + overall_recall) if (overall_precision + overall_recall) > 0 else 0.0

    overall = {"precision": overall_precision, "recall": overall_recall, "f1": overall_f1}

    return per_class, overall, y_true_for_cm, y_pred_for_cm


per_class_results, overall_results, y_true_cm, y_pred_cm = evaluate_precision_recall_f1(
    model, test_loader, device, iou_threshold=0.5, score_threshold=0.5
)

print("=== Overall (micro-averaged across all defect classes) ===")
print(f"Precision: {overall_results[\'precision\']:.4f}")
print(f"Recall:    {overall_results[\'recall\']:.4f}")
print(f"F1-Score:  {overall_results[\'f1\']:.4f}")

print("\\n=== Per-class ===")
for c, vals in per_class_results.items():
    print(f"{CLASS_NAMES[c]:>15}: Precision={vals[\'precision\']:.4f}  Recall={vals[\'recall\']:.4f}  F1={vals[\'f1\']:.4f}  (TP={vals[\'TP\']}, FP={vals[\'FP\']}, FN={vals[\'FN\']})")


In [ ]:
# --- Confusion matrix ---
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true_cm, y_pred_cm, labels=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("Ground Truth")
ax.set_title("Confusion Matrix (IoU >= 0.5)")

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")

fig.colorbar(im)
fig.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

print(cm)


## 9. Inference: "No defect" vs. defect type
This is the function you'll actually use to answer your thesis's required
output format: **does the image contain a defect, and if so, which type?**


In [ ]:
CONFIDENCE_THRESHOLD = 0.5  # tune this based on validation results

@torch.no_grad()
def predict_defect(model, image_path, class_names, device, threshold=CONFIDENCE_THRESHOLD):
    model.eval()
    img = Image.open(image_path).convert("RGB")
    img_tensor = F.to_tensor(img).to(device)

    output = model([img_tensor])[0]
    keep = output["scores"] >= threshold

    labels = output["labels"][keep].cpu().numpy()
    scores = output["scores"][keep].cpu().numpy()
    boxes = output["boxes"][keep].cpu().numpy()

    if len(labels) == 0:
        return {"result": "No defect", "detections": []}

    detections = [
        {"type": class_names[label], "confidence": float(score), "box": box.tolist()}
        for label, score, box in zip(labels, scores, boxes)
    ]
    # Report the highest-confidence defect type as the primary answer,
    # but keep all detections in case multiple defect types appear in one image.
    top = max(detections, key=lambda d: d["confidence"])
    return {"result": top["type"], "detections": detections}


# Example usage:
# result = predict_defect(model, "path/to/test_image.jpg", CLASS_NAMES, device)
# print(result)
# -> {'result': 'stringing', 'detections': [{'type': 'stringing', 'confidence': 0.87, 'box': [...]}]}


## 10. Auto-annotate a new (unlabeled) batch of images
If you get more images later, you don't have to draw every box from scratch again.
This cell runs your trained model on a folder of new, unlabeled images and writes a
Pascal VOC `.xml` file for each one — the same format Roboflow already uses.

**Workflow:**
1. Put your new raw images in a folder (e.g. `new_images/`)
2. Run this cell — it creates matching `.xml` files next to each image
3. Zip the folder and upload it to a new Roboflow project (drag-and-drop the images
   together with their `.xml` files — Roboflow auto-detects and pre-loads the boxes)
4. Review each image in Roboflow and fix anything the model got wrong or missed
   (correcting a box is much faster than drawing one from nothing)
5. Export and merge into your training set the same way you did before

This won't be perfect — it's exactly as good (and exactly as flawed) as your current
model, so double-check the classes it struggles with most (cracking, warping) more
carefully than the ones it's confident about (stringing).

Note: Roboflow's own custom-model upload for Label Assist currently only supports the
YOLO family (and a few others), not Faster R-CNN, so this XML-generation approach is the
practical path for your specific model.


In [ ]:
import xml.etree.ElementTree as ET
from xml.dom import minidom

def write_voc_xml(image_path, image_width, image_height, detections, output_path):
    annotation = ET.Element("annotation")

    ET.SubElement(annotation, "folder").text = os.path.basename(os.path.dirname(image_path))
    ET.SubElement(annotation, "filename").text = os.path.basename(image_path)

    size = ET.SubElement(annotation, "size")
    ET.SubElement(size, "width").text = str(image_width)
    ET.SubElement(size, "height").text = str(image_height)
    ET.SubElement(size, "depth").text = "3"

    for det in detections:
        obj = ET.SubElement(annotation, "object")
        ET.SubElement(obj, "name").text = det["type"]
        ET.SubElement(obj, "pose").text = "Unspecified"
        ET.SubElement(obj, "truncated").text = "0"
        ET.SubElement(obj, "difficult").text = "0"
        bndbox = ET.SubElement(obj, "bndbox")
        xmin, ymin, xmax, ymax = det["box"]
        ET.SubElement(bndbox, "xmin").text = str(int(xmin))
        ET.SubElement(bndbox, "ymin").text = str(int(ymin))
        ET.SubElement(bndbox, "xmax").text = str(int(xmax))
        ET.SubElement(bndbox, "ymax").text = str(int(ymax))

    xml_str = minidom.parseString(ET.tostring(annotation)).toprettyxml(indent="  ")
    with open(output_path, "w") as f:
        f.write(xml_str)


@torch.no_grad()
def auto_annotate_folder(model, image_folder, class_names, device, threshold=0.5):
    """
    Runs the trained model on every image in `image_folder` and writes a matching
    .xml file (Pascal VOC format) next to each image. Images with no detections
    above `threshold` get an .xml with no <object> tags (i.e. a "no defect" record).
    """
    model.eval()
    exts = (".jpg", ".jpeg", ".png")
    image_paths = [
        os.path.join(image_folder, f) for f in os.listdir(image_folder)
        if f.lower().endswith(exts)
    ]

    summary = {"annotated": 0, "no_defect": 0}
    for img_path in image_paths:
        img = Image.open(img_path).convert("RGB")
        width, height = img.size
        img_tensor = F.to_tensor(img).to(device)

        output = model([img_tensor])[0]
        keep = output["scores"] >= threshold

        labels = output["labels"][keep].cpu().numpy()
        scores = output["scores"][keep].cpu().numpy()
        boxes = output["boxes"][keep].cpu().numpy()

        detections = [
            {"type": class_names[label], "confidence": float(score), "box": box.tolist()}
            for label, score, box in zip(labels, scores, boxes)
        ]

        xml_path = os.path.splitext(img_path)[0] + ".xml"
        write_voc_xml(img_path, width, height, detections, xml_path)

        if detections:
            summary["annotated"] += 1
        else:
            summary["no_defect"] += 1

    return summary


# Example usage:
# summary = auto_annotate_folder(model, "new_images", CLASS_NAMES, device, threshold=0.5)
# print(summary)  # -> {'annotated': 42, 'no_defect': 8}
# Now zip "new_images" (images + generated .xml files together) and upload to Roboflow.


## Notes
- **What changed from the first run**: (1) training images now get random horizontal flips + mild
  brightness/contrast jitter each epoch, so the model sees more visual variety instead of the exact
  same 1190 images every time; (2) the model checkpoint is now saved whenever validation loss hits a
  new low, not just at the final epoch — so a late-training overfitting phase (like what happened
  before) no longer costs you your best result; (3) the learning rate now drops automatically when
  validation loss stalls, and training stops early if it stalls for 7 epochs straight, rather than
  running a fixed 20 epochs regardless of whether it's still helping.
- **Batch size / epochs**: batch size 4 and up to 30 epochs (with early stopping) are reasonable
  starting points for a Colab T4 GPU.
- **Class name mismatch**: if `evaluate_map` or training throws an error about labels, double-check
  the exact `<name>` strings in your `.xml` files match `CLASS_NAMES` exactly (case-sensitive,
  underscores vs. spaces, etc.) — you already hit this once with "layer shifting" vs "layer_shifting".
- **"No defect" validation gap**: since your dataset has no clean/no-defect images, this output
  path is only tested by the model *not* finding anything above threshold on a defect image —
  it hasn't been validated against genuinely clean prints. Worth flagging as a limitation, or add
  some no-defect images (see earlier discussion) if you have time.
- **If results are still weak on cracking/warping specifically**: those two classes had the lowest
  AP last time. If augmentation + best-checkpointing doesn't close the gap enough, the next lever
  to pull is checking whether those two classes have noticeably fewer or lower-quality annotations
  than stringing/layer shifting — class imbalance and box tightness matter more than model choice
  at this scale of dataset.
- **Metric reporting**: if you later compare your results with a published study, make sure the
  (e.g., precision/recall/F1 at IoU 0.5, or mAP@0.5 only), let me know and I'll adjust the
  evaluation cell to report the same numbers in the same way.


## 11. GitHub / reproducibility notes
- This notebook is intended to be the main reproducible entry point for the project.
- The dataset itself is not included in this repository by default. Check the dataset license before redistributing any images or annotations.
- Do not commit API keys, passwords, personal Google Drive files, or private credentials.
- The current reported metrics are **preliminary** until the dataset split has been independently verified and clean/no-defect specimens are added and evaluated.
- For final thesis experiments, keep the test set completely unseen during training and model selection.